# Notebook 5 — Dynamic Pricing Engine & Revenue Simulation

### Build a demand-responsive hotel pricing model and simulate its potential revenue impact using historical booking data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style="whitegrid")

df = pd.read_csv(r"../data/processed/hotel_bookings_cleaned.csv")

df['arrival_date'] = pd.to_datetime(df['arrival_date'])
df['departure_date'] = pd.to_datetime(df['departure_date'])

df = df.drop_duplicates().copy()

print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

Dataset shape: (87396, 49)
Duplicate rows: 0


## 1. Daily Demand Modeling

Reconstruct daily occupied rooms from completed stays and create a daily calendar for each hotel.

In [2]:
stayed = df.loc[
    (df['is_canceled'] == 0) &
    (df['stay_duration'] > 0),
    ['hotel', 'arrival_date', 'stay_duration']
].copy()

stayed['occupied_date'] = [
    pd.date_range(
        start=arrival,
        periods=int(duration),
        freq='D'
    )
    for arrival, duration in zip(
        stayed['arrival_date'],
        stayed['stay_duration']
    )
]

daily_occupied = (
    stayed[['hotel', 'occupied_date']]
    .explode('occupied_date')
    .groupby(['hotel', 'occupied_date'])
    .size()
    .reset_index(name='occupied_rooms')
)

daily_occupied['occupied_date'] = pd.to_datetime(
    daily_occupied['occupied_date']
)

daily_occupied.head()

,hotel,occupied_date,occupied_rooms
0,City Hotel,2015-07-01,6
1,City Hotel,2015-07-02,7
2,City Hotel,2015-07-03,8
3,City Hotel,2015-07-04,12
4,City Hotel,2015-07-05,7


In [3]:
# Create a Complete Daily Calender
calendar_parts = []

for hotel, group in df.groupby('hotel'):
    start_date = group['arrival_date'].min()
    end_date = group['departure_date'].max()

    dates = pd.date_range(
        start=start_date,
        end=end_date - pd.Timedelta(days=1),
        freq='D'
    )

    calendar_parts.append(
        pd.DataFrame({
            'hotel': hotel,
            'date': dates
        })
    )

daily_calendar = pd.concat(
    calendar_parts,
    ignore_index=True
)

daily_calendar = daily_calendar.drop_duplicates(
    subset=['hotel', 'date']
).sort_values(['hotel', 'date']).reset_index(drop=True)

daily_demand_model = daily_calendar.merge(
    daily_occupied.rename(
        columns={'occupied_date': 'date'}
    ),
    on=['hotel', 'date'],
    how='left'
)

daily_demand_model['occupied_rooms'] = (
    daily_demand_model['occupied_rooms']
    .fillna(0)
    .astype(int)
)

print("Duplicate hotel-date rows:",
      daily_demand_model.duplicated(['hotel', 'date']).sum())

daily_demand_model.head()

Duplicate hotel-date rows: 0


,hotel,date,occupied_rooms
0,City Hotel,2015-07-01,6
1,City Hotel,2015-07-02,7
2,City Hotel,2015-07-03,8
3,City Hotel,2015-07-04,12
4,City Hotel,2015-07-05,7


In [4]:
# Estimate Capacity and Occupancy
capacity_estimate = (
    daily_occupied
    .groupby('hotel')['occupied_rooms']
    .quantile(0.95)
    .round()
    .astype(int)
    .reset_index(name='capacity_proxy')
)

daily_demand_model = daily_demand_model.merge(
    capacity_estimate,
    on='hotel',
    how='left'
)

daily_demand_model['occupancy_rate'] = (
    daily_demand_model['occupied_rooms'] /
    daily_demand_model['capacity_proxy'] * 100
).round(2)

capacity_estimate

,hotel,capacity_proxy
0,City Hotel,205
1,Resort Hotel,179


## 2. Booking Pace

Compare how many reservations had been made by different lead-time checkpoints. Historical median booking counts provide the reference pace for each hotel.

In [5]:
checkpoints = [90, 60, 30, 14, 7, 1, 0]

checkpoint_weights = {
    90: 0.05,
    60: 0.10,
    30: 0.15,
    14: 0.20,
    7: 0.20,
    1: 0.15,
    0: 0.15
}
booking_by_lead = (
    df[df['is_canceled'] == 0]
    .groupby(['hotel', 'arrival_date', 'lead_time'])
    .size()
    .reset_index(name='bookings_made')
)

pace_records = []

for (hotel, arrival_date), group in booking_by_lead.groupby(
    ['hotel', 'arrival_date']
):
    for checkpoint in checkpoints:
        cumulative = group.loc[
            group['lead_time'] >= checkpoint,
            'bookings_made'
        ].sum()

        pace_records.append({
            'hotel': hotel,
            'arrival_date': arrival_date,
            'days_before_arrival': checkpoint,
            'cumulative_bookings': cumulative
        })

booking_checkpoints = pd.DataFrame(pace_records)

pace_benchmark = (
    booking_checkpoints
    .groupby(['hotel', 'days_before_arrival'])
    ['cumulative_bookings']
    .median()
    .reset_index(name='median_bookings')
)

booking_checkpoints = booking_checkpoints.merge(
    pace_benchmark,
    on=['hotel', 'days_before_arrival'],
    how='left'
)

booking_checkpoints['pace_ratio'] = np.where(
    booking_checkpoints['median_bookings'] > 0,
    booking_checkpoints['cumulative_bookings'] /
    booking_checkpoints['median_bookings'],
    1.0
)

booking_checkpoints['weighted_pace'] = (
    booking_checkpoints['pace_ratio'] *
    booking_checkpoints['days_before_arrival'].map(
        checkpoint_weights
    )
)

daily_pace = (
    booking_checkpoints
    .groupby(['hotel', 'arrival_date'], as_index=False)
    .agg(booking_pace_score=('weighted_pace', 'sum'))
)

daily_pace['booking_pace_score'] = (
    daily_pace['booking_pace_score'].round(2)
)

daily_pace['bounded_pace_score'] = (
    daily_pace['booking_pace_score'].clip(0, 2)
)

print("Duplicate pace rows:",
      daily_pace.duplicated(['hotel', 'arrival_date']).sum())

daily_pace.head()

Duplicate pace rows: 0


,hotel,arrival_date,booking_pace_score,bounded_pace_score
0,City Hotel,2015-07-01,0.18,0.18
1,City Hotel,2015-07-02,0.01,0.01
2,City Hotel,2015-07-03,0.11,0.11
3,City Hotel,2015-07-04,0.12,0.12
4,City Hotel,2015-07-06,0.03,0.03


In [6]:
# Merge booking pace and add calender features
daily_pace_for_merge = daily_pace.rename(
    columns={'arrival_date': 'date'}
)

daily_demand_model = daily_demand_model.merge(
    daily_pace_for_merge[
        ['hotel', 'date', 'booking_pace_score',
         'bounded_pace_score']
    ],
    on=['hotel', 'date'],
    how='left'
)

daily_demand_model['pace_observed'] = (
    daily_demand_model['booking_pace_score'].notna()
)

daily_demand_model['bounded_pace_score'] = (
    daily_demand_model['bounded_pace_score'].fillna(1.0)
)

daily_demand_model['month_number'] = (
    daily_demand_model['date'].dt.month
)

daily_demand_model['month_name'] = (
    daily_demand_model['date'].dt.month_name()
)

daily_demand_model['day_name'] = (
    daily_demand_model['date'].dt.day_name()
)

daily_demand_model['is_weekend'] = (
    daily_demand_model['date'].dt.dayofweek >= 5
)

daily_demand_model['season'] = np.select(
    [
        daily_demand_model['month_number'].isin([12, 1, 2]),
        daily_demand_model['month_number'].isin([3, 4, 5]),
        daily_demand_model['month_number'].isin([6, 7, 8]),
        daily_demand_model['month_number'].isin([9, 10, 11])
    ],
    ['Winter', 'Spring', 'Summer', 'Autumn'],
    default='Unknown'
)

daily_demand_model.head()

,hotel,date,occupied_rooms,capacity_proxy,occupancy_rate,booking_pace_score,bounded_pace_score,pace_observed,month_number,month_name,day_name,is_weekend,season
0,City Hotel,2015-07-01,6,205,2.93,0.18,0.18,True,7,July,Wednesday,False,Summer
1,City Hotel,2015-07-02,7,205,3.41,0.01,0.01,True,7,July,Thursday,False,Summer
2,City Hotel,2015-07-03,8,205,3.90,0.11,0.11,True,7,July,Friday,False,Summer
3,City Hotel,2015-07-04,12,205,5.85,0.12,0.12,True,7,July,Saturday,True,Summer
4,City Hotel,2015-07-05,7,205,3.41,NaN,1.00,False,7,July,Sunday,True,Summer


## 3. Seasonality

Measure each hotel's seasonal occupancy relative to its own overall average.

In [7]:
season_summary = (
    daily_demand_model
    .groupby(['hotel', 'season'], as_index=False)
    .agg(average_occupancy=('occupancy_rate', 'mean'))
)

hotel_occupancy_baseline = (
    daily_demand_model
    .groupby('hotel', as_index=False)
    .agg(
        overall_average_occupancy=('occupancy_rate', 'mean')
    )
)

season_summary = season_summary.merge(
    hotel_occupancy_baseline,
    on='hotel',
    how='left'
)

season_summary['seasonality_index'] = (
    season_summary['average_occupancy'] /
    season_summary['overall_average_occupancy']
).round(2)

daily_demand_model = daily_demand_model.merge(
    season_summary[
        ['hotel', 'season', 'seasonality_index']
    ],
    on=['hotel', 'season'],
    how='left'
)

season_summary

,hotel,season,average_occupancy,overall_average_occupancy,seasonality_index
0,City Hotel,Autumn,63.32,67.93,0.93
1,City Hotel,Spring,82.87,67.93,1.22
2,City Hotel,Summer,72.50,67.93,1.07
3,City Hotel,Winter,51.44,67.93,0.76
4,Resort Hotel,Autumn,73.97,75.68,0.98
5,Resort Hotel,Spring,77.76,75.68,1.03
6,Resort Hotel,Summer,94.98,75.68,1.26
7,Resort Hotel,Winter,49.16,75.68,0.65


## 4. Combined Demand Score

Combine occupancy, booking pace, seasonality, and weekend status into a single demand score. The weights and thresholds are modeling assumptions.

In [8]:
daily_demand_model['occupancy_score'] = (
    daily_demand_model['occupancy_rate'].clip(0, 100)
)

daily_demand_model['pace_score'] = (
    daily_demand_model['bounded_pace_score'] * 50
).clip(0, 100)

daily_demand_model['seasonality_score'] = (
    (
        daily_demand_model['seasonality_index']
        .clip(0.5, 1.5) - 0.5
    ) * 100
)

daily_demand_model['weekend_score'] = np.where(
    daily_demand_model['is_weekend'],
    60,
    40
)

daily_demand_model['demand_score'] = (
    0.45 * daily_demand_model['occupancy_score'] +
    0.30 * daily_demand_model['pace_score'] +
    0.20 * daily_demand_model['seasonality_score'] +
    0.05 * daily_demand_model['weekend_score']
).round(2)

daily_demand_model['demand_level'] = np.select(
    [
        daily_demand_model['demand_score'] < 40,
        daily_demand_model['demand_score'] < 60,
        daily_demand_model['demand_score'] < 80
    ],
    ['Low', 'Normal', 'High'],
    default='Very High'
)

daily_demand_model[
    ['hotel', 'date', 'occupancy_rate',
     'booking_pace_score', 'pace_observed',
     'seasonality_index', 'demand_score', 'demand_level']
].head()

,hotel,date,occupancy_rate,booking_pace_score,pace_observed,seasonality_index,demand_score,demand_level
0,City Hotel,2015-07-01,2.93,0.18,True,1.07,17.42,Low
1,City Hotel,2015-07-02,3.41,0.01,True,1.07,15.08,Low
2,City Hotel,2015-07-03,3.90,0.11,True,1.07,16.80,Low
3,City Hotel,2015-07-04,5.85,0.12,True,1.07,18.83,Low
4,City Hotel,2015-07-05,3.41,NaN,False,1.07,30.93,Low


## 5. Dynamic Pricing Engine

Use each hotel's median historical ADR as the baseline and adjust it according to the demand level.

In [9]:
valid_adr = df[
    (df['is_canceled'] == 0) &
    (df['adr'] > 0) &
    np.isfinite(df['adr'])
]

hotel_baseline_adr = (
    valid_adr
    .groupby('hotel')['adr']
    .median()
    .round(2)
    .reset_index(name='baseline_adr')
)

daily_demand_model = daily_demand_model.merge(
    hotel_baseline_adr,
    on='hotel',
    how='left'
)

price_adjustments = {
    'Low': 0.90,
    'Normal': 1.00,
    'High': 1.10,
    'Very High': 1.20
}

daily_demand_model['price_multiplier'] = (
    daily_demand_model['demand_level']
    .map(price_adjustments)
)

daily_demand_model['recommended_adr'] = (
    daily_demand_model['baseline_adr'] *
    daily_demand_model['price_multiplier']
).round(2)

daily_demand_model[
    ['hotel', 'date', 'demand_score',
     'demand_level', 'baseline_adr',
     'price_multiplier', 'recommended_adr']
].head()

,hotel,date,demand_score,demand_level,baseline_adr,price_multiplier,recommended_adr
0,City Hotel,2015-07-01,17.42,Low,104.00,0.90,93.60
1,City Hotel,2015-07-02,15.08,Low,104.00,0.90,93.60
2,City Hotel,2015-07-03,16.80,Low,104.00,0.90,93.60
3,City Hotel,2015-07-04,18.83,Low,104.00,0.90,93.60
4,City Hotel,2015-07-05,30.93,Low,104.00,0.90,93.60


## 6. Price-Response Scenarios

Since true price elasticity cannot be estimated from the available data, test three hypothetical levels of customer price sensitivity.

In [10]:
# Defining Scenarios 
price_elasticity_scenarios = {
    'Conservative': 0.3,
    'Moderate': 0.6,
    'Aggressive': 0.9
}

In [11]:
# Preparing simulation data
simulation = daily_demand_model[
    [
        'hotel', 'date', 'season', 'demand_score',
        'demand_level', 'occupied_rooms',
        'capacity_proxy', 'baseline_adr',
        'recommended_adr', 'price_multiplier'
    ]
].copy()

# Apply the same capacity constraint to both strategies.
simulation['baseline_rooms'] = np.minimum(
    simulation['occupied_rooms'],
    simulation['capacity_proxy']
)

simulation['baseline_revenue'] = (
    simulation['baseline_rooms'] *
    simulation['baseline_adr']
)

simulation.head()

,hotel,date,season,demand_score,demand_level,occupied_rooms,capacity_proxy,baseline_adr,recommended_adr,price_multiplier,baseline_rooms,baseline_revenue
0,City Hotel,2015-07-01,Summer,17.42,Low,6,205,104.00,93.60,0.90,6,624.00
1,City Hotel,2015-07-02,Summer,15.08,Low,7,205,104.00,93.60,0.90,7,728.00
2,City Hotel,2015-07-03,Summer,16.80,Low,8,205,104.00,93.60,0.90,8,832.00
3,City Hotel,2015-07-04,Summer,18.83,Low,12,205,104.00,93.60,0.90,12,1248.00
4,City Hotel,2015-07-05,Summer,30.93,Low,7,205,104.00,93.60,0.90,7,728.00


In [12]:
# Simulate all scenarios
scenario_results = []

for scenario, elasticity in price_elasticity_scenarios.items():
    scenario_df = simulation.copy()

    scenario_df['demand_multiplier'] = (
        scenario_df['price_multiplier'] ** (-elasticity)
    )

    scenario_df['expected_rooms'] = (
        scenario_df['baseline_rooms'] *
        scenario_df['demand_multiplier']
    )

    scenario_df['expected_rooms'] = np.minimum(
        scenario_df['expected_rooms'],
        scenario_df['capacity_proxy']
    )

    scenario_df['dynamic_revenue'] = (
        scenario_df['expected_rooms'] *
        scenario_df['recommended_adr']
    )

    scenario_df['scenario'] = scenario
    scenario_results.append(scenario_df)

simulation_all = pd.concat(
    scenario_results,
    ignore_index=True
)

simulation_all.head()

,hotel,date,season,demand_score,demand_level,occupied_rooms,capacity_proxy,baseline_adr,recommended_adr,price_multiplier,baseline_rooms,baseline_revenue,demand_multiplier,expected_rooms,dynamic_revenue,scenario
0,City Hotel,2015-07-01,Summer,17.42,Low,6,205,104.00,93.60,0.90,6,624.00,1.03,6.19,579.63,Conservative
1,City Hotel,2015-07-02,Summer,15.08,Low,7,205,104.00,93.60,0.90,7,728.00,1.03,7.22,676.24,Conservative
2,City Hotel,2015-07-03,Summer,16.80,Low,8,205,104.00,93.60,0.90,8,832.00,1.03,8.26,772.85,Conservative
3,City Hotel,2015-07-04,Summer,18.83,Low,12,205,104.00,93.60,0.90,12,1248.00,1.03,12.39,1159.27,Conservative
4,City Hotel,2015-07-05,Summer,30.93,Low,7,205,104.00,93.60,0.90,7,728.00,1.03,7.22,676.24,Conservative


## 7. Revenue Simulation

Compare baseline pricing with demand-responsive pricing under each price-sensitivity scenario.

In [13]:
scenario_summary = (
    simulation_all
    .groupby('scenario', as_index=False)
    .agg(
        baseline_revenue=('baseline_revenue', 'sum'),
        dynamic_revenue=('dynamic_revenue', 'sum'),
        baseline_rooms=('baseline_rooms', 'sum'),
        expected_rooms=('expected_rooms', 'sum'),
        capacity_room_nights=('capacity_proxy', 'sum')
    )
)

scenario_summary['revenue_change'] = (
    scenario_summary['dynamic_revenue'] -
    scenario_summary['baseline_revenue']
)

scenario_summary['revenue_change_pct'] = (
    scenario_summary['revenue_change'] /
    scenario_summary['baseline_revenue'] * 100
).round(2)

scenario_summary['baseline_utilization_pct'] = (
    scenario_summary['baseline_rooms'] /
    scenario_summary['capacity_room_nights'] * 100
).round(2)

scenario_summary['dynamic_utilization_pct'] = (
    scenario_summary['expected_rooms'] /
    scenario_summary['capacity_room_nights'] * 100
).round(2)

scenario_summary.round(2)

,scenario,baseline_revenue,dynamic_revenue,baseline_rooms,expected_rooms,capacity_room_nights,revenue_change,revenue_change_pct,baseline_utilization_pct,dynamic_utilization_pct
0,Aggressive,19793803.00,19945697.64,220753,206576.04,308889,151894.64,0.77,71.47,66.88
1,Conservative,19793803.00,20910598.30,220753,215790.58,308889,1116795.30,5.64,71.47,69.86
2,Moderate,19793803.00,20416479.43,220753,211067.85,308889,622676.43,3.15,71.47,68.33


In [14]:
# Hotel level results
hotel_scenario_summary = (
    simulation_all
    .groupby(['scenario', 'hotel'], as_index=False)
    .agg(
        baseline_revenue=('baseline_revenue', 'sum'),
        dynamic_revenue=('dynamic_revenue', 'sum'),
        baseline_rooms=('baseline_rooms', 'sum'),
        expected_rooms=('expected_rooms', 'sum'),
        capacity_room_nights=('capacity_proxy', 'sum')
    )
)

hotel_scenario_summary['revenue_change_pct'] = (
    (
        hotel_scenario_summary['dynamic_revenue'] -
        hotel_scenario_summary['baseline_revenue']
    ) /
    hotel_scenario_summary['baseline_revenue'] * 100
).round(2)

hotel_scenario_summary['baseline_utilization_pct'] = (
    hotel_scenario_summary['baseline_rooms'] /
    hotel_scenario_summary['capacity_room_nights'] * 100
).round(2)

hotel_scenario_summary['dynamic_utilization_pct'] = (
    hotel_scenario_summary['expected_rooms'] /
    hotel_scenario_summary['capacity_room_nights'] * 100
).round(2)

hotel_scenario_summary.round(2)

,scenario,hotel,baseline_revenue,dynamic_revenue,baseline_rooms,expected_rooms,capacity_room_nights,revenue_change_pct,baseline_utilization_pct,dynamic_utilization_pct
0,Aggressive,City Hotel,11609728.00,11694794.70,111632,104855.47,164615,0.73,67.81,63.70
1,Aggressive,Resort Hotel,8184075.00,8250902.93,109121,101720.57,144274,0.82,75.63,70.51
2,Conservative,City Hotel,11609728.00,12235510.27,111632,109258.05,164615,5.39,67.81,66.37
3,Conservative,Resort Hotel,8184075.00,8675088.03,109121,106532.53,144274,6.00,75.63,73.84
4,Moderate,City Hotel,11609728.00,11958547.27,111632,107000.58,164615,3.00,67.81,65.00
5,Moderate,Resort Hotel,8184075.00,8457932.16,109121,104067.27,144274,3.35,75.63,72.13


In [15]:
# Seasonal results
season_scenario_summary = (
    simulation_all
    .groupby(['scenario', 'hotel', 'season'], as_index=False)
    .agg(
        baseline_revenue=('baseline_revenue', 'sum'),
        dynamic_revenue=('dynamic_revenue', 'sum'),
        baseline_rooms=('baseline_rooms', 'sum'),
        expected_rooms=('expected_rooms', 'sum')
    )
)

season_scenario_summary['revenue_change_pct'] = (
    (
        season_scenario_summary['dynamic_revenue'] -
        season_scenario_summary['baseline_revenue']
    ) /
    season_scenario_summary['baseline_revenue'].replace(0, np.nan)
    * 100
).round(2)

season_scenario_summary.round(2).head()

,scenario,hotel,season,baseline_revenue,dynamic_revenue,baseline_rooms,expected_rooms,revenue_change_pct
0,Aggressive,City Hotel,Autumn,2591056.00,2602019.92,24914,24040.14,0.42
1,Aggressive,City Hotel,Spring,3250000.00,3286404.02,31250,28293.71,1.12
2,Aggressive,City Hotel,Summer,3783728.00,3825654.51,36382,33025.91,1.11
3,Aggressive,City Hotel,Winter,1984944.00,1980716.26,19086,19495.71,-0.21
4,Aggressive,Resort Hotel,Autumn,1936125.00,1950732.94,25815,24171.15,0.75


In [16]:
# Demand level results
demand_level_summary = (
    simulation_all
    .groupby(['scenario', 'hotel', 'demand_level'], as_index=False)
    .agg(
        days=('date', 'count'),
        baseline_revenue=('baseline_revenue', 'sum'),
        dynamic_revenue=('dynamic_revenue', 'sum'),
        baseline_rooms=('baseline_rooms', 'sum'),
        expected_rooms=('expected_rooms', 'sum')
    )
)

demand_level_summary['revenue_change_pct'] = (
    (
        demand_level_summary['dynamic_revenue'] -
        demand_level_summary['baseline_revenue']
    ) /
    demand_level_summary['baseline_revenue'].replace(0, np.nan)
    * 100
).round(2)

demand_level_summary.round(2).head()

,scenario,hotel,demand_level,days,baseline_revenue,dynamic_revenue,baseline_rooms,expected_rooms,revenue_change_pct
0,Aggressive,City Hotel,High,326,5842408.00,5898358.30,56177,51559.08,0.96
1,Aggressive,City Hotel,Low,188,1129960.00,1118117.18,10865,11945.70,-1.05
2,Aggressive,City Hotel,Normal,179,2411240.00,2411240.00,23185,23185.00,0.00
3,Aggressive,City Hotel,Very High,110,2226120.00,2267079.22,21405,18165.70,1.84
4,Aggressive,Resort Hotel,High,356,4214850.00,4255213.86,56198,51578.35,0.96


## 8. Model Validation

Check that the pricing output has unique hotel-date rows, valid prices, and expected room counts within the capacity proxy.

In [17]:
assert not daily_demand_model.duplicated(
    ['hotel', 'date']
).any(), "Duplicate hotel-date rows found."

assert daily_demand_model['recommended_adr'].notna().all(), \
    "Missing recommended ADR values."

assert (daily_demand_model['recommended_adr'] > 0).all(), \
    "Non-positive recommended ADR found."

assert (
    simulation_all['expected_rooms'] <=
    simulation_all['capacity_proxy'] + 1e-9
).all(), "Expected rooms exceed capacity."

assert simulation_all['expected_rooms'].ge(0).all(), \
    "Negative expected room count found."

print("All validation checks passed.")

All validation checks passed.


In [18]:
# Final Pricing Output
pricing_output = daily_demand_model[
    [
        'hotel', 'date', 'occupancy_rate',
        'booking_pace_score', 'pace_observed',
        'season', 'seasonality_index',
        'demand_score', 'demand_level',
        'baseline_adr', 'recommended_adr'
    ]
].copy()

pricing_output = pricing_output.sort_values(
    ['hotel', 'date']
).reset_index(drop=True)

pricing_output.head(20)

,hotel,date,occupancy_rate,booking_pace_score,pace_observed,season,seasonality_index,demand_score,demand_level,baseline_adr,recommended_adr
0,City Hotel,2015-07-01,2.93,0.18,True,Summer,1.07,17.42,Low,104.00,93.60
1,City Hotel,2015-07-02,3.41,0.01,True,Summer,1.07,15.08,Low,104.00,93.60
2,City Hotel,2015-07-03,3.90,0.11,True,Summer,1.07,16.80,Low,104.00,93.60
3,City Hotel,2015-07-04,5.85,0.12,True,Summer,1.07,18.83,Low,104.00,93.60
4,City Hotel,2015-07-05,3.41,NaN,False,Summer,1.07,30.93,Low,104.00,93.60
5,City Hotel,2015-07-06,2.93,0.03,True,Summer,1.07,15.17,Low,104.00,93.60
6,City Hotel,2015-07-07,3.41,0.05,True,Summer,1.07,15.68,Low,104.00,93.60
7,City Hotel,2015-07-08,4.88,0.20,True,Summer,1.07,18.60,Low,104.00,93.60
8,City Hotel,2015-07-09,4.88,0.03,True,Summer,1.07,16.05,Low,104.00,93.60
9,City Hotel,2015-07-10,2.93,0.06,True,Summer,1.07,15.62,Low,104.00,93.60


## 9. Business Recommendations

- Adjust prices according to demand conditions rather than applying one fixed increase throughout the year.
- Use hotel-specific seasonality when planning prices.
- Apply conservative increases when demand is uncertain or customers may be price-sensitive.
- Review the simulated results separately for City Hotel and Resort Hotel.
- Treat the revenue uplift as a scenario estimate, not a guaranteed forecast.

The model uses a historical occupancy-based capacity proxy and assumed price elasticity. Actual inventory, competitor prices, and rejected bookings are unavailable, so results should be validated against real hotel operations before implementation.

In [ ]:
pricing_output = (
    daily_demand_model[
        [
            'hotel',
            'date',
            'occupied_rooms',
            'capacity_proxy',
            'occupancy_rate',
            'booking_pace_score',
            'pace_observed',
            'season',
            'seasonality_index',
            'demand_score',
            'demand_level',
            'baseline_adr',
            'price_multiplier',
            'recommended_adr'
        ]
    ]
    .sort_values(['hotel', 'date'])
    .reset_index(drop=True)
)

pricing_output.to_csv(
    "../data/processed/hotel_daily_pricing.csv",
    index=False
)

print("Pricing rows:", len(pricing_output))
print("Duplicate hotel-date rows:",
      pricing_output.duplicated(['hotel', 'date']).sum())

pricing_output.head()

Pricing rows: 1609
Duplicate hotel-date rows: 0


,hotel,date,occupied_rooms,capacity_proxy,occupancy_rate,booking_pace_score,pace_observed,season,seasonality_index,demand_score,demand_level,baseline_adr,price_multiplier,recommended_adr
0,City Hotel,2015-07-01,6,205,2.93,0.18,True,Summer,1.07,17.42,Low,104.00,0.90,93.60
1,City Hotel,2015-07-02,7,205,3.41,0.01,True,Summer,1.07,15.08,Low,104.00,0.90,93.60
2,City Hotel,2015-07-03,8,205,3.90,0.11,True,Summer,1.07,16.80,Low,104.00,0.90,93.60
3,City Hotel,2015-07-04,12,205,5.85,0.12,True,Summer,1.07,18.83,Low,104.00,0.90,93.60
4,City Hotel,2015-07-05,7,205,3.41,NaN,False,Summer,1.07,30.93,Low,104.00,0.90,93.60


In [ ]:
scenario_output = simulation_all[
    [
        'scenario',
        'hotel',
        'date',
        'season',
        'demand_level',
        'baseline_adr',
        'recommended_adr',
        'price_multiplier',
        'demand_multiplier',
        'baseline_rooms',
        'expected_rooms',
        'capacity_proxy',
        'baseline_revenue',
        'dynamic_revenue'
    ]
].copy()

scenario_output['revenue_change'] = (
    scenario_output['dynamic_revenue'] -
    scenario_output['baseline_revenue']
)

scenario_output.to_csv(
    "../data/processed/hotel_pricing_scenarios.csv",
    index=False
)

print("Scenario rows:", len(scenario_output))
scenario_output.head()

Scenario rows: 4827


,scenario,hotel,date,season,demand_level,baseline_adr,recommended_adr,price_multiplier,demand_multiplier,baseline_rooms,expected_rooms,capacity_proxy,baseline_revenue,dynamic_revenue,revenue_change
0,Conservative,City Hotel,2015-07-01,Summer,Low,104.00,93.60,0.90,1.03,6,6.19,205,624.00,579.63,-44.37
1,Conservative,City Hotel,2015-07-02,Summer,Low,104.00,93.60,0.90,1.03,7,7.22,205,728.00,676.24,-51.76
2,Conservative,City Hotel,2015-07-03,Summer,Low,104.00,93.60,0.90,1.03,8,8.26,205,832.00,772.85,-59.15
3,Conservative,City Hotel,2015-07-04,Summer,Low,104.00,93.60,0.90,1.03,12,12.39,205,1248.00,1159.27,-88.73
4,Conservative,City Hotel,2015-07-05,Summer,Low,104.00,93.60,0.90,1.03,7,7.22,205,728.00,676.24,-51.76


In [ ]:
scenario_summary.to_csv(
    "../data/processed/hotel_scenario_summary.csv",
    index=False
)
scenario_summary

,scenario,baseline_revenue,dynamic_revenue,baseline_rooms,expected_rooms,capacity_room_nights,revenue_change,revenue_change_pct,baseline_utilization_pct,dynamic_utilization_pct
0,Aggressive,19793803.00,19945697.64,220753,206576.04,308889,151894.64,0.77,71.47,66.88
1,Conservative,19793803.00,20910598.30,220753,215790.58,308889,1116795.30,5.64,71.47,69.86
2,Moderate,19793803.00,20416479.43,220753,211067.85,308889,622676.43,3.15,71.47,68.33
